# Teste isolado — ABCR (melhoresrodovias.org.br)

Fonte candidata: **ABCR - Associação Brasileira de Concessionárias de
Rodovias**, setor Transporte. Já existia placeholder em `controle_fontes`
(`nome_fonte='ABCR'`, `source_id='—'`, `status='Não iniciada'`,
`importancia_original='Baixa'`). Notebook **descartável** (Fase 1) — sem
dispatcher, sem `atualizar_status_fonte`, sem gravar nada em produção.

URL: `https://melhoresrodovias.org.br/noticias/`.

**Nome da fonte**: em todo lugar (`controle_fontes`, comentários, etc.),
usar só **"ABCR"** — não "ABCR Notícias", sem a palavra "notícias" no
nome em lugar nenhum.

**Nota sobre `robots.txt`**: o arquivo tem um bloco Cloudflare que
desautoriza nominalmente vários bots de IA (`ClaudeBot`, `GPTBot`,
`Google-Extended`, etc.), mas o grupo genérico `User-agent: *` permite
tudo (`Allow: /`, também confirmado no bloco Yoast duplicado
`Disallow:` vazio). Decisão explícita do usuário: prosseguir normalmente
— o scraper usa User-Agent de navegador (mesma convenção já usada em
todas as outras fontes do projeto), não se identifica como ClaudeBot, e
o grupo `*` cobre esse caso.

In [ ]:
%pip install --quiet httpx beautifulsoup4 lxml
dbutils.library.restartPython()

In [ ]:
import urllib.parse
import httpx
from bs4 import BeautifulSoup

BASE = "https://melhoresrodovias.org.br"
URL_NOTICIAS = f"{BASE}/noticias/"

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
)
HEADERS = {"User-Agent": USER_AGENT, "Accept-Language": "pt-BR,pt;q=0.9"}

## Teste 1 — robots.txt e a listagem estática

Confere o `robots.txt` (ver nota acima) e se a listagem em
`/noticias/` já vem com título/resumo/link prontos no HTML estático,
sem precisar de JS para o primeiro lote.

In [ ]:
with httpx.Client(headers=HEADERS, timeout=30, follow_redirects=True) as client:
    resp_robots = client.get(f"{BASE}/robots.txt")
    resp_noticias = client.get(URL_NOTICIAS)

print(f"/robots.txt -> HTTP {resp_robots.status_code} ({len(resp_robots.text)} chars)")
print(f"/noticias/  -> HTTP {resp_noticias.status_code} ({len(resp_noticias.text)} chars)")

soup_listagem = BeautifulSoup(resp_noticias.text, "lxml")
artigos = soup_listagem.find_all("article")
print(f"\n<article> encontrados no HTML estático: {len(artigos)}")

## Teste 2 — extrair título, resumo e link de cada item

Estrutura confirmada (Elementor / Essential Addons "eael-grid-post"):
título+link em `h2.eael-entry-title a`, resumo em
`.eael-grid-post-excerpt p`.

In [ ]:
itens = []
for artigo in artigos:
    tag_titulo = artigo.select_one("h2.eael-entry-title a")
    if not tag_titulo:
        continue
    titulo = tag_titulo.get_text(" ", strip=True)
    url = urllib.parse.urljoin(URL_NOTICIAS, tag_titulo["href"].strip())
    tag_resumo = artigo.select_one(".eael-grid-post-excerpt p")
    resumo = tag_resumo.get_text(" ", strip=True) if tag_resumo else None
    itens.append({"titulo": titulo, "url": url, "resumo": resumo})

print(f"{len(itens)} itens extraídos.\n")
for item in itens:
    print(f"- {item['titulo']}")
    print(f"  {item['url']}")
    print(f"  resumo: {(item['resumo'] or '')[:120]}\n")

## Teste 3 — paginação ("Carregar mais")

A listagem usa um botão "Carregar mais" via AJAX (widget
`eael-load-more` do Elementor, endpoint `admin-ajax.php`). Testado se
existe um padrão de URL alternativo `/noticias/page/N/` que devolva
conteúdo distinto da primeira página.

In [ ]:
with httpx.Client(headers=HEADERS, timeout=30, follow_redirects=True) as client:
    resp_p2 = client.get(f"{BASE}/noticias/page/2/")

print(f"/noticias/page/2/ -> HTTP {resp_p2.status_code} ({len(resp_p2.text)} chars)")

soup_p2 = BeautifulSoup(resp_p2.text, "lxml")
links_p1 = {i["url"] for i in itens}
links_p2 = {
    urllib.parse.urljoin(URL_NOTICIAS, a["href"].strip())
    for a in soup_p2.select("h2.eael-entry-title a")
}
print(f"itens na page/2/: {len(links_p2)}")
print(f"sobreposição com a página 1: {len(links_p1 & links_p2)} de {len(links_p1)}")
print(
    "\n=> page/2/ NÃO é paginação real (mesmo conteúdo da página 1) -- "
    "AJAX é o único mecanismo. Por instrução do usuário, tudo bem capturar "
    "só o primeiro lote (4 itens) nesse caso."
    if links_p1 and links_p1 == (links_p1 & links_p2)
    else "\n=> page/2/ parece devolver itens distintos, revisar antes de decidir."
)

## Teste 4 — abrir uma notícia e extrair texto completo + data

Página do artigo: data em `<time>` (dois `<time>` na página — o
primeiro é a data por extenso, ex. "agosto 12, 2026", o segundo é o
horário), texto completo em
`.elementor-widget-theme-post-content`.

In [ ]:
with httpx.Client(headers=HEADERS, timeout=30, follow_redirects=True) as client:
    resp_artigo = client.get(itens[0]["url"])

soup_artigo = BeautifulSoup(resp_artigo.text, "lxml")

tags_time = soup_artigo.find_all("time")
print(f"<time> encontrados: {[t.get_text(strip=True) for t in tags_time]}")

conteudo = soup_artigo.select_one(".elementor-widget-theme-post-content")
print(f"\n.elementor-widget-theme-post-content encontrado: {conteudo is not None}")
if conteudo:
    texto = conteudo.get_text("\n", strip=True)
    print(f"tamanho do texto: {len(texto)} chars\n")
    print(texto[:600])

## Conclusão da Fase 1

`robots.txt` permite o `User-agent: *` genérico (ver nota no topo sobre
a desautorização nominal de bots de IA, decisão explícita de prosseguir
mesmo assim). A listagem estática (`/noticias/`) traz o primeiro lote
(4 itens) com título, resumo e link prontos, sem precisar de JS. A
paginação ("Carregar mais") depende de AJAX e `/noticias/page/N/` não é
paginação real (devolve o mesmo conteúdo da página 1) — captura limitada
ao primeiro lote, conforme instrução ("se não conseguir paginar, tudo
bem capturar só a primeira leva"). A página de cada notícia tem texto
completo limpo em `.elementor-widget-theme-post-content`, com data em
`<time>` (dois elementos: data por extenso + horário) — sem paywall, sem
WAF, sem Selenium.

**Avaliação para a Fase 2**: encaixa no dispatcher genérico
`ingest-scraping` — é "baixar uma URL e extrair itens de forma padrão",
listagem via HTML estático simples (`baixar_pagina()` genérico + parsing
próprio de `listar_abcr()`), texto via `SELETORES_CONTEUDO` genérico
adicionando `.elementor-widget-theme-post-content`.

Nada gravado em `controle_fontes` — teste isolado, decisão de integração
pendente.